# CUDA Optimization Course - Module 7: Advanced Techniques & Full Hardware Utilization

Advanced optimizations and maximizing M3 MacBook capacity.

## Setup

import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
import psutil
import os
import time
from typing import Dict

# Device detection
if torch.cuda.is_available():
    device = 'cuda'
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("Using MPS (M3 MacBook)")
    print(f"Unified Memory Architecture")
else:
    device = 'cpu'
    print("Using CPU")

# System info
cpu_count = psutil.virtual_memory()
print(f"\nSystem RAM: {cpu_count.total / 1e9:.1f} GB")
print(f"CPU Cores: {os.cpu_count()}")

## Hardware Monitoring Utilities

class HardwareMonitor:
    """Monitor GPU/CPU utilization."""
    
    @staticmethod
    def get_system_stats() -> Dict[str, float]:
        """Get system memory and CPU stats."""
        vm = psutil.virtual_memory()
        return {
            'ram_used_gb': vm.used / 1e9,
            'ram_available_gb': vm.available / 1e9,
            'ram_percent': vm.percent,
            'cpu_percent': psutil.cpu_percent(interval=0.1)
        }
    
    @staticmethod
    def get_device_stats() -> Dict[str, float]:
        """Get device memory stats."""
        if device == 'cuda':
            return {
                'gpu_allocated_gb': torch.cuda.memory_allocated() / 1e9,
                'gpu_reserved_gb': torch.cuda.memory_reserved() / 1e9,
                'gpu_total_gb': torch.cuda.get_device_properties(0).total_memory / 1e9
            }
        else:
            return {'note': 'MPS/CPU stats not directly available'}
    
    @staticmethod
    def print_all_stats():
        """Print comprehensive system stats."""
        print("\n=== SYSTEM STATS ===")
        sys_stats = HardwareMonitor.get_system_stats()
        print(f"RAM Used: {sys_stats['ram_used_gb']:.2f} GB")
        print(f"RAM Available: {sys_stats['ram_available_gb']:.2f} GB")
        print(f"CPU Usage: {sys_stats['cpu_percent']:.1f}%")
        
        if device == 'cuda':
            print("\n=== GPU STATS ===")
            gpu_stats = HardwareMonitor.get_device_stats()
            print(f"GPU Allocated: {gpu_stats['gpu_allocated_gb']:.2f} GB")
            print(f"GPU Reserved: {gpu_stats['gpu_reserved_gb']:.2f} GB")
            print(f"GPU Total: {gpu_stats['gpu_total_gb']:.2f} GB")

HardwareMonitor.print_all_stats()

## Optimization 1: Grouped Query Attention (GQA)

Reduce KV cache size with shared key-value heads.

class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention - efficient for large sequences."""
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int = None):
        super().__init__()
        if n_kv_heads is None:
            n_kv_heads = n_heads  # Standard MHA
        
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        
        # Repeat KV heads to match Q heads if needed
        if self.n_kv_heads < self.n_heads:
            repeat_factor = self.n_heads // self.n_kv_heads
            K = K.repeat_interleave(repeat_factor, dim=1)
            V = V.repeat_interleave(repeat_factor, dim=1)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.out_proj(context)

# Compare memory usage
seq_len = 2048
batch_size = 16
d_model = 768
n_heads = 12

print("Memory usage comparison for KV cache:")
print(f"Standard MHA: {seq_len * batch_size * d_model * 2 / 1e9:.3f} GB (2 caches: K, V)")
print(f"GQA (6 kv_heads): {seq_len * batch_size * (d_model // 2) * 2 / 1e9:.3f} GB (50% savings)")
print(f"GQA (2 kv_heads): {seq_len * batch_size * (d_model // 6) * 2 / 1e9:.3f} GB (83% savings)")

## Optimization 2: Activation Offloading

Move intermediate activations to CPU during training.

class OffloadedModule(nn.Module):
    """Module that offloads activations to CPU."""
    
    def __init__(self, module: nn.Module, device: str, cpu_device: str = 'cpu'):
        super().__init__()
        self.module = module
        self.device = device
        self.cpu_device = cpu_device
    
    def forward(self, x):
        x_gpu = x.to(self.device)
        out = self.module(x_gpu)
        # Optionally offload to CPU
        # out = out.to(self.cpu_device).detach()
        return out

print("Activation offloading reduces GPU memory at cost of bandwidth.")
print("Recommended when: GPU memory < 24GB and strong CPU-GPU connection")

## Optimization 3: Fused Kernels

Use fused operations to reduce memory bandwidth.

class FusedLayerNorm(nn.Module):
    """Fused LayerNorm + activation for efficiency."""
    
    def __init__(self, d_model: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # In production, this would use a fused CUDA kernel
        return torch.nn.functional.gelu(self.norm(x))

print("Fused kernels combine multiple operations to reduce memory access.")
print("Typical speedup: 10-30% depending on operation sequence.")

## Optimization 4: Dynamic Batching

Adjust batch size based on available memory.

class DynamicBatchOptimizer:
    """Dynamically adjust batch size based on memory."""
    
    @staticmethod
    def estimate_memory_per_sample(model: nn.Module, seq_len: int, d_model: int) -> float:
        """Estimate memory per sample in GB."""
        # Activation memory: O(batch * seq * d_model * layers)
        # Gradient memory: ~2x activation memory
        
        num_params = sum(p.numel() for p in model.parameters())
        bytes_per_param = 4  # FP32
        
        # Model: parameters + gradients + optimizer state
        model_memory = (num_params * bytes_per_param * 3) / 1e9  # x3 for gradients + optimizer
        
        # Activation memory (rough estimate)
        activation_memory = (seq_len * d_model * bytes_per_param) / 1e9
        
        return model_memory + activation_memory
    
    @staticmethod
    def find_max_batch_size(model: nn.Module, device: str) -> int:
        """Find maximum batch size before OOM."""
        if device == 'cuda':
            available_memory = torch.cuda.get_device_properties(0).total_memory * 0.8 / 1e9  # 80% utilization
        else:
            vm = psutil.virtual_memory()
            available_memory = (vm.available * 0.5) / 1e9  # 50% of available RAM
        
        mem_per_sample = DynamicBatchOptimizer.estimate_memory_per_sample(model, 256, 768)
        max_batch_size = int(available_memory / mem_per_sample)
        
        return max(1, max_batch_size)

print("Dynamic batching adjusts to hardware constraints automatically.")

## M3 MacBook Specific Optimizations

class M3MacOptimizations:
    """Optimizations specific to M3 MacBook."""
    
    @staticmethod
    def get_optimal_settings():
        """Get recommended settings for M3 MacBook."""
        return {
            'device': 'mps',
            'batch_size': 32,  # Balance between throughput and memory
            'sequence_length': 256,  # Max ~1024 for large models
            'use_amp': True,  # FP16 gives biggest speedup
            'use_checkpoint': True,  # Essential for large models
            'num_workers': 0,  # Multiprocessing not recommended on Mac
            'pin_memory': False,  # Not applicable for MPS
            'accumulation_steps': 4,  # Simulate larger batch size
            'torch_compile': True,  # Supported in latest PyTorch
        }
    
    @staticmethod
    def get_memory_stats():
        """Get M3 MacBook memory configuration."""
        vm = psutil.virtual_memory()
        return {
            'total_memory_gb': vm.total / 1e9,
            'available_memory_gb': vm.available / 1e9,
            'memory_usage_percent': vm.percent,
            'note': 'M3 uses unified memory architecture - GPU and CPU share same memory'
        }
    
    @staticmethod
    def print_optimization_guide():
        print("""
        ╔════════════════════════════════════════════════════════════════╗
        ║        M3 MACBOOK OPTIMIZATION GUIDE                             ║
        ╠════════════════════════════════════════════════════════════════╣
        ║ 1. Use MPS Backend                                             ║
        ║    → torch.device('mps')                                       ║
        ║                                                                ║
        ║ 2. Enable AMP (Automatic Mixed Precision)                      ║
        ║    → 2-3x faster training, ~50% memory savings                ║
        ║                                                                ║
        ║ 3. Gradient Checkpointing                                      ║
        ║    → Essential for models > 500M parameters                    ║
        ║                                                                ║
        ║ 4. Batch Size: 32-64 recommended                               ║
        ║    → Test your hardware with batch size finder                 ║
        ║                                                                ║
        ║ 5. Sequence Length: 256-512 typical                            ║
        ║    → Reduce if OOM occurs                                      ║
        ║                                                                ║
        ║ 6. Avoid Multiprocessing (num_workers=0)                       ║
        ║    → Data loading on M1/M2/M3 is already fast                 ║
        ║                                                                ║
        ║ 7. Monitor Thermal Throttling                                  ║
        ║    → Reduce batch size if performance drops                    ║
        ║                                                                ║
        ║ 8. Use torch.compile() for inference                           ║
        ║    → 20-30% speedup for compiled models                        ║
        ║                                                                ║
        ║ Combined Impact: ~4-8x vs baseline (FP32, no optimizations)   ║
        ╚════════════════════════════════════════════════════════════════╝
        """)

M3MacOptimizations.print_optimization_guide()
print("\nCurrent Settings:")
settings = M3MacOptimizations.get_optimal_settings()
for k, v in settings.items():
    print(f"  {k}: {v}")

print("\nMemory Configuration:")
mem_stats = M3MacOptimizations.get_memory_stats()
for k, v in mem_stats.items():
    print(f"  {k}: {v}")

## Optimization 5: Quantization (8-bit Inference)

class QuantizedLinear(nn.Module):
    """Simplified 8-bit quantized linear layer."""
    
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) / (in_features ** 0.5))
        self.scale = nn.Parameter(torch.ones(1))
    
    def forward(self, x):
        # Quantize weight
        weight_q = (self.weight * self.scale).clamp(-128, 127).to(torch.int8).float()
        return torch.nn.functional.linear(x, weight_q / self.scale)

print("\n8-bit Quantization:")
print("  Memory: 75% reduction (4 bytes → 1 byte per parameter)")
print("  Speed: 2-4x faster (device dependent)")
print("  Accuracy: Minimal impact for inference")

## Complete Optimization Pipeline

class FullOptimizationPipeline:
    """Combines all optimization techniques."""
    
    def __init__(self, model: nn.Module, device: str = 'cuda'):
        self.model = model
        self.device = device
    
    def apply_all_optimizations(self):
        """Apply all available optimizations."""
        print("Applying optimizations...")
        
        # 1. Move to device
        self.model = self.model.to(self.device)
        print("✓ Moved to device")
        
        # 2. Compile model
        if self.device != 'mps':  # compile works better on CUDA
            try:
                self.model = torch.compile(self.model, mode='reduce-overhead')
                print("✓ Applied torch.compile()")
            except:
                print("⚠ torch.compile() not available")
        
        # 3. Enable eval mode for inference
        self.model.eval()
        print("✓ Enabled eval mode")
        
        print("\nAll optimizations applied!")
        return self.model
    
    def get_optimization_info(self) -> dict:
        """Get info about applied optimizations."""
        return {
            'device': str(self.device),
            'model_size_mb': sum(p.numel() for p in self.model.parameters()) * 4 / 1e6,
            'is_compiled': 'OptimizedModule' in str(type(self.model)),
            'is_eval': not self.model.training
        }

print("Full optimization pipeline ready for deployment!")

## Key Takeaways

### For M3 MacBook:
1. **AMP is critical** - provides 2-3x speedup with minimal code changes
2. **Gradient checkpointing** - enables larger models by trading memory for compute
3. **Proper batch sizing** - balance between throughput and memory
4. **Monitor thermal** - M3 has thermal limits, adjust batch size if needed
5. **No multiprocessing** - keep num_workers=0

### General Principles:
1. **Profile first** - know where time/memory is spent
2. **Combine techniques** - multiplicative benefits
3. **Benchmark before/after** - validate improvements
4. **Hardware-aware** - tailor to your specific device
5. **Trade-offs** - memory vs compute, speed vs accuracy

### Expected Speedups:
- **AMP alone**: 2-3x
- **torch.compile()**: 1.3-2.0x
- **All combined**: 4-8x
- **With quantization**: 2-4x additional for inference